# MAGMa Fragmentation Deep Dive

**Molecule:** 2-Bromo-N-cyclohexylacetamide  
**SMILES:** `O=C(CBr)NC1CCCCC1`  
**mol_id:** 176980 | **InChIKey:** VQKRFCPNGGNHNN-UHFFFAOYSA-N | **RI:** SemiStdNP=1493

Questions answered:
1. **Ring systems** — how cyclohexane ring bonds can be broken
2. **Isotopes** — `IsotopePatternCalculator` vs `DifferentiableIsotopePatternModule`, and which is used when
3. **Fragment selection** — what `max_nodes` does and which fragments survive
4. **Predicted intensity** — what the GNN actually does in full-enumeration mode
5. **Same-mass collision** — which fragment wins at a shared integer m/z
6. **Full gallery** — all fragments at every BFS depth

In [ ]:
%load_ext autoreload
%autoreload 2

from collections import Counter, defaultdict

import matplotlib.pyplot as plt
import numpy as np
from IPython.display import display, HTML
from rdkit.Chem import Draw
from rdkit.Chem.Draw import rdMolDraw2D

from icicle.data.fragmentation_engine import (
    FragmentEngine,
    FragmentationParams,
    extend,
)
from icicle.data.isotope_distribution import IsotopePatternCalculator
from icicle.utils.visualization.style import set_style

set_style("manuscript")
SMILES = "O=C(CBr)NC1CCCCC1"

---
## 0. Build the engine & inspect atom ordering

The engine canonicalises via InChI (`SMILES → InChI → mol`), so atom indices may differ from the input SMILES.

In [ ]:
engine = FragmentEngine(
    SMILES, FragmentationParams(max_tree_depth=3, num_h_shifts=1)
)
engine.generate_fragments()

drawer = rdMolDraw2D.MolDraw2DSVG(520, 300)
drawer.drawOptions().addAtomIndices = True
drawer.DrawMolecule(engine.mol)
drawer.FinishDrawing()
display(HTML(drawer.GetDrawingText()))

print("idx  sym  implicit_H  mass+H")
for i, sym in enumerate(engine.atom_symbols):
    print(
        f"  {i:2d}  {sym:2s}   {engine.atom_hs[i]}        {engine.atom_weights_h[i]:.4f}"
    )
print(
    f"\nHeavy atoms: {engine.natoms}  Total H: {engine.total_hs}  Full mass: {engine.full_weight:.4f}"
)
print(f"Unique fragments after BFS: {len(engine.frag_to_entry)}")


def bitmask_atoms(bitmask, eng):
    return [
        (i, eng.atom_symbols[i])
        for i in range(eng.natoms)
        if bitmask & (1 << i)
    ]

---
## 1. Ring system fragmentation

The engine works by **atom removal**, never explicit bond breaking.  
When atom `i` is removed the critical question is: does the remaining graph split?

```
remove_atom(fragment, i):
    template = fragment ^ (1 << i)         # flip atom i off
    neighbors = atoms bonded to i, still in template

    if len(neighbors) == 1:                # linear chain end: one piece
        return [template]

    if len(neighbors) >= 2:                # branch point OR ring atom
        extended = []
        for each neighbor n:
            if n already in any component in extended:   # is_ring check
                skip — DFS already covered it via the ring
            else:
                component = DFS(n, template)             # flood-fill
                extended.append(component)
        return extended
```

**Ring rule:** A ring atom has two neighbors that both survive in `template`.  
DFS from the *first* neighbor walks the full ring and already includes the *second* neighbor.  
So `is_ring` fires for the second → **no split, ring stays intact**.  
Removing one ring atom costs 1 bond-weight unit and yields one connected fragment.

You need to remove **two consecutive ring atoms** (2 BFS depth levels) before the ring opens.

In [ ]:
ring_info = engine.mol.GetRingInfo()
ring_atoms = list(ring_info.AtomRings()[0])
print("Cyclohexane ring atom indices:", ring_atoms)

root_frag = engine.get_root_frag()
ex = ring_atoms[0]
print(
    f"\nStep 1 — remove ring atom {ex} ({engine.atom_symbols[ex]}) from root:"
)
step1 = engine.remove_atom(root_frag, ex)
print(f"  Fragments produced: {len(step1)}  (ring intact → expected 1)")
for fi in step1:
    s = engine.atom_pass_stats(fi.new_frag)
    print(
        f"  → {s['form']}  mass={s['base_mass']:.3f}  bond_cost={fi.rm_bond_t}"
    )

print(f"\nStep 2 — remove a second adjacent ring atom:")
frag1 = step1[0].new_frag
second = next(a for a in ring_atoms if a != ex and frag1 & (1 << a))
step2 = engine.remove_atom(frag1, second)
print(f"  Fragments produced: {len(step2)}")
for fi in step2:
    s = engine.atom_pass_stats(fi.new_frag)
    print(
        f"  → {s['form']}  mass={s['base_mass']:.3f}  atoms={[a for a in range(engine.natoms) if fi.new_frag & (1 << a)]}"
    )

print(
    "\n→ Still 1 fragment even after two removals? That means the ring fully broke but"
)
print(
    "  the non-ring chain kept the two pieces connected. Inspect atoms above."
)

---
## 2. Isotopes: `IsotopePatternCalculator` vs `DifferentiableIsotopePatternModule`

There are two separate implementations. The key surprise: **`IsotopePatternCalculator` is NOT used at inference**.  
Look at what `EIMSPredictorFromFullEnumeration._generate_fragments()` passes to the engine:

```python
# eims_predictor.py line 590
FragmentationParams(
    ....
    detect_isotope_patterns=False,   # ← explicitly disabled!
)
```

So at inference the BFS engine runs with **no isotope expansion** — `get_frag_masses()` produces
only monoisotopic bins `{0: 1.0}` per fragment.  The isotope expansion happens **inside the GNN**
via `DifferentiableIsotopePatternModule`.

| | `IsotopePatternCalculator` | `DifferentiableIsotopePatternModule` |
|---|---|---|
| **Location** | `fragmentation_engine.py` | inside `IntensityPredictor` (`intensity_model.py`) |
| **Type** | Plain Python / NumPy | PyTorch `nn.Module` |
| **Used at inference** | **No** — `detect_isotope_patterns=False` | **Yes** — runs inside GNN forward pass |
| **Used during training** | No | Yes — gradients flow through isotope weights |
| **Used for MAGMa labelling** | Yes — the standalone BFS with isotopes enabled | No |
| **Algorithm** | Binomial (2-isotope) or approx (multi-isotope), Python loops | Batched binomial in torch tensors on GPU |
| **Output** | `Dict[iso_shift → rel_abundance]` | Tensors `(masses, intensities, batch_indices)` |

**Short answer:** the `IsotopePatternCalculator` in the engine is for the MAGMa labelling pipeline
(creating ground-truth DAGs from experimental spectra), not for inference.
The notebook you're reading has isotopes enabled only so we can study it — in real inference it's off.

In [ ]:
# Show what isotope patterns look like for key formulas in this molecule
calc = IsotopePatternCalculator(min_intensity_threshold=0.01)

formulas = [
    ("CH2Br", "CH2Br — 1× Br"),
    ("C6H11", "C6H11 — cyclohexyl"),
    ("C2H3BrNO", "C2H3BrNO — acyl+Br"),
    ("C8H14BrNO", "C8H14BrNO — full mol"),
]

fig, axes = plt.subplots(1, 4, figsize=(6.5, 2.5))
for ax, (formula, label) in zip(axes, formulas):
    d = calc.calculate_isotope_distribution(calc.parse_formula(formula))
    shifts, abunds = zip(*sorted(d.items()))
    ax.bar(shifts, abunds, width=0.6)
    for s, a in zip(shifts, abunds):
        ax.text(s, a + 0.04, f"{a:.2f}", ha="center", fontsize=6)
    ax.set_xticks(list(shifts))
    ax.set_xlabel("iso shift (Da)", fontweight="bold")
    ax.set_ylim(0, 1.3)
    ax.set_title(label, fontsize=7, fontweight="bold")
    ax.grid(True, linestyle="--", alpha=0.4)
axes[0].set_ylabel("Rel. abundance", fontweight="bold")
plt.suptitle(
    "IsotopePatternCalculator — Br forces near-equal M and M+2",
    fontweight="bold",
)
plt.tight_layout()
plt.savefig("isotope_patterns.svg", bbox_inches="tight")
# plt.savefig("isotope_patterns.png", dpi=300, bbox_inches="tight")
plt.show()

# Confirm inference engine has isotopes disabled
engine_infer = FragmentEngine(
    SMILES,
    FragmentationParams(
        max_tree_depth=3, num_h_shifts=1, detect_isotope_patterns=False
    ),
)
engine_infer.generate_fragments()
fh_inf, _, sh_inf, m_inf, _ = engine_infer.get_frag_masses()
print(f"With detect_isotope_patterns=False (inference mode):")
print(f"  entries: {len(m_inf)}   unique bins: {len(set(m_inf))}")
print(f"With detect_isotope_patterns=True  (labelling mode):")
fh_lab, _, sh_lab, m_lab, _ = engine.get_frag_masses()
print(f"  entries: {len(m_lab)}   unique bins: {len(set(m_lab))}")
print("\n→ Isotope expansion doubles entries for Br-bearing fragments.")
print(
    "  At inference this is handled by DifferentiableIsotopePatternModule inside the GNN."
)

---
## 3. Fragment selection: what does `max_nodes` do?

The BFS generates **all** fragments up to `max_tree_depth=3` and `max_broken_bonds=6`, unconditionally.  
For this molecule that is 66 fragments. Larger / more complex molecules can produce hundreds.

When `max_nodes` is set, the code **prunes after the full BFS**:

```python
# eims_predictor.py lines 597-606
if max_nodes is not None and len(fragment_engine.frag_to_entry) > max_nodes:
    top_entries = sorted(
        fragment_engine.frag_to_entry.items(),
        key=lambda kv: (kv[1].tree_depth, kv[1].max_broken),  # shallowest first
    )[:max_nodes]
    fragment_engine.frag_to_entry = dict(top_entries)
```

**Selection rule:** sort by `(tree_depth, max_broken)` ascending, keep the top-`max_nodes`.  
This keeps **shallowest, least-broken fragments first** — i.e., fragments that required the fewest
bond breaks to produce, which are considered most chemically reliable.

In [ ]:
# Simulate max_nodes trimming and show which fragments survive vs are dropped
MAX_NODES = 20

all_entries = list(engine.frag_to_entry.items())
sorted_entries = sorted(
    all_entries, key=lambda kv: (kv[1].tree_depth, kv[1].max_broken)
)

kept = sorted_entries[:MAX_NODES]
dropped = sorted_entries[MAX_NODES:]

print(f"Total fragments: {len(all_entries)}")
print(f"After max_nodes={MAX_NODES}: keep {len(kept)}, drop {len(dropped)}")

print(
    f"\n{'rank':>5}  {'depth':>6}  {'max_broken':>11}  {'formula':>20}  {'mass':>8}  status"
)
print("-" * 70)
for rank, (h, e) in enumerate(sorted_entries):
    status = "KEPT" if rank < MAX_NODES else "dropped"
    marker = "←" if rank == MAX_NODES - 1 else ""
    print(
        f"{rank:>5}  {e.tree_depth:>6}  {e.max_broken:>11}  {e.form:>20}  {e.base_mass:>8.2f}  {status} {marker}"
    )

# Show depth distribution of kept vs dropped
kept_depths = Counter(e.tree_depth for _, e in kept)
drop_depths = Counter(e.tree_depth for _, e in dropped)
print("\nDepth distribution:")
print(f"  {'depth':>6}  {'kept':>6}  {'dropped':>8}")
for d in sorted(set(kept_depths) | set(drop_depths)):
    print(f"  {d:>6}  {kept_depths.get(d, 0):>6}  {drop_depths.get(d, 0):>8}")

---
## 4. Predicted intensity in full-enumeration mode

**"Full enumeration"** = how fragments are *generated* (MAGMa BFS). Intensities still come from the GNN.

```
SMILES
  │
  ▼ FragmentEngine.generate_fragments()       MAGMa BFS, detect_isotope_patterns=False
  │  → 66 structural fragments (bitmasks)
  │  [if max_nodes: prune by (depth, broken_bonds) ascending]
  │
  ▼ TreeProcessor.featurize_frag()            bitmask → DGL graph per fragment
  │  nodes = heavy atoms present in bitmask
  │  edges = bonds between present atoms, bond-type features
  │
  ▼ IntensityPredictor.predict_intensities()  GNN forward pass (one call, batched)
  │  GatedGNN → Set Transformer → MLP
  │  input:  66 fragment graphs + root graph (context) + broken_bonds + masses
  │  output: one scalar per (fragment, h_shift) pair
  │
  ▼ DifferentiableIsotopePatternModule        expand each scalar across iso bins
  │  {M+0: intensity × 1.0,  M+2: intensity × 0.97}  for Br-bearing fragments
  │  fully differentiable — gradients flow during training
  │
  ▼ scatter-add onto 750-bin spectrum         multiple entries can hit same bin → SUM
  ▼ normalise → predicted spectrum [0..1]
```

**Without a checkpoint** (this notebook) we can only inspect the BFS structure.  
The cells below show how the fragments map onto spectrum bins structurally.

In [ ]:
# Inference-mode engine (isotopes off, matches EIMSPredictorFromFullEnumeration)
fhashes, _, shifts, masses, scores = engine_infer.get_frag_masses()

bin_counts = Counter(masses)
collisions = {mz: cnt for mz, cnt in bin_counts.items() if cnt > 1}

print("--- Inference-mode engine (detect_isotope_patterns=False) ---")
print(f"Structural fragments:  {len(engine_infer.frag_to_entry)}")
print(f"(frag, h_shift) entries:  {len(masses)}")
print(f"Unique integer m/z bins:  {len(bin_counts)}")
print(f"Bins with ≥2 contributors (h-shift collisions): {len(collisions)}")

print("\n→ Collisions here come only from H-shifts, not isotopes.")
print("  The GNN sees one (frag, h_shift) entry per row — one scalar out.")
print(
    "  DifferentiableIsotopePatternModule then spreads each scalar to M+0, M+2, etc."
)

fig, ax = plt.subplots(figsize=(4, 2.5))
hist = Counter(bin_counts.values())
xs, ys = zip(*sorted(hist.items()))
ax.bar(xs, ys)
ax.set_xlabel("# (frag, h_shift) entries per m/z bin", fontweight="bold")
ax.set_ylabel("# bins", fontweight="bold")
ax.set_title(
    "Inference-mode bin collision distribution\n(H-shift collisions only)",
    fontweight="bold",
)
ax.grid(True, linestyle="--", alpha=0.4)
plt.tight_layout()
plt.savefig("bin_collisions_infer.png", dpi=150, bbox_inches="tight")
plt.show()

---
## 5. Same-mass collision: which fragment hash is kept?

In `frags_to_intens()` — used only for **visualisation**, not for the loss:

```python
if mass already occupied:
    if new_intensity > current_intensity:
        current["frag_hash"] = new_hash    # label = highest-intensity contributor
    current["inten"] += new_intensity       # value  = SUM of all contributors
```

The spectrum bin value is always a sum. The hash label is the dominant contributor,
used only for the fragment hover display.

In [ ]:
# Use the labelling-mode engine (isotopes on) to show interesting Br collisions
fh_lab, _, sh_lab, m_lab, _ = engine.get_frag_masses()
bin_lab = Counter(m_lab)
coll_lab = {mz: cnt for mz, cnt in bin_lab.items() if cnt > 1}
top5 = sorted(coll_lab.items(), key=lambda x: -x[1])[:5]

print(f"{'m/z':>6}  {'#entries':>8}  contributors")
print("-" * 80)
calc = IsotopePatternCalculator(min_intensity_threshold=0.01)
for mz, cnt in top5:
    mask = m_lab == mz
    entries = []
    for fh, sh in zip(fh_lab[mask], sh_lab[mask]):
        fe = engine.frag_to_entry[fh]
        iso = calc.calculate_isotope_distribution(calc.parse_formula(fe.form))
        for iso_s, abund in iso.items():
            if (
                abund >= 0.01
                and round(fe.base_mass + sh * 1.007825 + iso_s) == mz
            ):
                entries.append(f"{fe.form}(h={sh:+d},+{iso_s}Da)")
    print(f"{mz:>6}  {cnt:>8}  {', '.join(entries[:5])}")

# Visualise fragments at most-contested bin
target_mz = top5[0][0]
competing = list(dict.fromkeys(fh_lab[m_lab == target_mz]))
n = min(len(competing), 4)
fig, axes = plt.subplots(1, n, figsize=(min(n * 1.8, 6.5), 2.5))
if n == 1:
    axes = [axes]
for ax, fh in zip(axes, competing[:n]):
    fe = engine.frag_to_entry[fh]
    di = engine.get_draw_dict(fe.frag)
    img = Draw.MolToImage(
        di.mol,
        highlightAtoms=di.hatoms,
        highlightBonds=di.hbonds,
        size=(200, 160),
    )
    ax.imshow(img)
    iso = calc.calculate_isotope_distribution(calc.parse_formula(fe.form))
    via = []
    for sh in range(-fe.max_remove_hs, fe.max_add_hs + 1):
        for iso_s in iso:
            if (
                iso[iso_s] >= 0.01
                and round(fe.base_mass + sh * 1.007825 + iso_s) == target_mz
            ):
                via.append(f"h={sh:+d},+{iso_s}Da")
    ax.set_title(
        f"{fe.form}\n{fe.base_mass:.2f} Da  d={fe.tree_depth}",
        fontsize=7,
        fontweight="bold",
    )
    ax.set_xlabel(", ".join(via), fontsize=6)
    ax.axis("off")
plt.suptitle(f"Fragments colliding at m/z = {target_mz}", fontweight="bold")
plt.tight_layout()
plt.savefig("collision_frags.png", dpi=150, bbox_inches="tight")
plt.show()

---
## 6. Full fragment gallery — all depths

Every unique fragment the BFS found, grouped by depth (0 = root, 3 = deepest).  
Blue overlay = atoms **present** in the fragment.  
`*` = fragment reachable from multiple parent paths (same structure, different atom-removal orders).

In [ ]:
depth_entries = defaultdict(list)
for h, entry in engine.frag_to_entry.items():
    depth_entries[entry.tree_depth].append((h, entry))
for d in depth_entries:
    depth_entries[d].sort(key=lambda x: -x[1].base_mass)  # heaviest first

for depth in sorted(depth_entries):
    frags_d = depth_entries[depth]
    n = len(frags_d)
    cols = min(n, 6)
    rows = (n + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 1.6, rows * 2.2))
    axes_flat = np.array(axes).flatten() if n > 1 else np.array([axes])

    for ax, (h, entry) in zip(axes_flat, frags_d):
        di = engine.get_draw_dict(entry.frag)
        img = Draw.MolToImage(
            di.mol,
            highlightAtoms=di.hatoms,
            highlightBonds=di.hbonds,
            size=(200, 160),
        )
        ax.imshow(img)
        multi = "*" if len(entry.parent_hashes) > 1 else ""
        ax.set_title(
            f"{entry.form}{multi}\n{entry.base_mass:.1f} Da",
            fontsize=7,
            fontweight="bold",
        )
        ax.axis("off")

    for ax in axes_flat[n:]:
        ax.axis("off")

    plt.suptitle(
        f"Depth {depth} — {n} fragment(s)  (* = multiple parent paths)",
        fontweight="bold",
        fontsize=9,
    )
    plt.tight_layout()
    plt.savefig(f"all_frags_depth{depth}.svg", bbox_inches="tight")
    plt.savefig(f"all_frags_depth{depth}.png", dpi=150, bbox_inches="tight")
    plt.show()

In [ ]:
print(
    f"{'Depth':>6}  {'#frags':>7}  {'mass range':>22}  {'Br frags':>9}  {'multi-parent':>13}"
)
print("-" * 66)
for d in sorted(depth_entries):
    frags_d = depth_entries[d]
    masses_d = [e.base_mass for _, e in frags_d]
    n_br = sum(1 for _, e in frags_d if "Br" in e.form)
    n_mp = sum(1 for _, e in frags_d if len(e.parent_hashes) > 1)
    print(
        f"{d:>6}  {len(frags_d):>7}  {min(masses_d):>9.2f} – {max(masses_d):>9.2f}  {n_br:>9}  {n_mp:>13}"
    )

---
## 7. MAGMa labelling pipeline: how ground-truth training targets are built

This runs **once offline** (via `label_ground_truth_dags.py`) and produces `magma_tree.hdf5`.  
The `FragmentSpecDataset` then reads this at training time alongside `spectra.hdf5`.

### Full pipeline

```
INPUT: experimental EI spectrum (mz, intensity pairs)   +   SMILES
       from spectra.hdf5                                    from metadata.tsv

1. FRAGMENT ENUMERATION   (_create_fragment_engine)
   FragmentEngine(SMILES, detect_isotope_patterns=True)   ← isotopes ON here
   .generate_fragments()
   get_frag_masses()  →  (frag_hashes, frag_inds, shift_inds, masses, scores)
   scores = broken-bond cost per (frag, h_shift, iso_shift) entry

2. SORT BY SCORE   (_analyze_spectrum line 447)
   sort_idx = argsort(scores)    ← lower score = fewer/cheaper bond breaks
   All arrays reordered: shallowest fragments come first.
   This is the tiebreak — among all entries that match an observed m/z,
   the cheapest fragmentation path wins.

3. MASS MATCHING   (_unit_mass_comparison)
   For every observed peak at mz_obs:
       peak_mask[obs] = min |round(masses) - round(mz_obs)| < 1
   Boolean: was this peak explained by ANY (frag, h_shift, iso) entry?

4. CANDIDATE SELECTION PER PEAK   (_generate_tsv_output lines 612-623)
   For each matched peak:
       matched_entries = all entries within ±0.5 Da of observed m/z
       min_score       = min(scores[matched_entries])
       best_entries    = matched_entries where score == min_score   (all ties kept)
       best_entries    = best_entries[:5]                           (cap at 5)

   ⚠ KEY: if two structurally different fragments have the same min score,
   BOTH are stored. The DAG records all candidates; the GNN learns which matters.
   Stored in magma_tsv.hdf5: mz_observed, inten, frag_hashes (comma-sep), h_shift, formula

5. TREE BUILDING   (_build_and_prune_tree)
   Seed set = all matched fragment hashes ("leaves").
   Walk parent_hashes backwards → add all ancestors up to root.
   Result: leaves + every intermediate fragment needed to connect them to the root.

6. GREEDY PRUNING   (_greedy_prune)
   Keep only a minimal connected subtree covering all observed leaves:
   - Bottom-up, one depth level at a time
   - Greedily select the parent that covers the most uncovered children
   - Tiebreak: lowest node_score (cheapest bond path)

7. TREE STRUCTURE OUTPUT   (_create_tree_structure)
   Per pruned fragment node, stored in magma_tree.hdf5[mol_id] as JSON:
     frag          : bitmask (which atoms are present)
     is_observed   : True if directly matched to an experimental peak
     atoms_pulled  : which atoms were removed to produce this fragment from its parent
     parents       : parent fragment hashes
     base_mass     : monoisotopic mass
     intens        : np.zeros(...)    ← ALWAYS ZERO — not stored here!
     max_broken, tree_depth, max_remove_hs, max_add_hs

OUTPUT: magma_tree.hdf5  — graph structure, which fragments were observed
        magma_tsv.hdf5   — per-peak fragment assignments with scores and shifts
```

### Where do intensity targets come from?

**The DAG stores no intensities.** Training targets are the raw experimental spectrum,  
loaded live at `__getitem__` time by `FragmentSpecDataset`:

```python
# dags.py lines 551-558
mz, intensities = self._load_spectrum(mol_id)           # from spectra.hdf5
binned = self.binner(mz, intensities)["spectrum"]       # bin into 750 1-Da bins
result = {
    "inten_targs": binned,          # ← THE SUPERVISION TARGET
    **processed_tree["dgl_tree"],   # ← fragment DGL graphs + masses (from MAGMa tree)
}
```

The loss (`_common_step` in `intensity_model.py`) compares `output_binned` (GNN prediction)  
vs `inten_targs` (binned experimental) using cosine / JS-divergence / entropy similarity.

### How intensity is distributed across fragments (the key question)

**The GNN is never given per-fragment intensity labels.**  
It only receives the fragment graph structure + root context + broken-bond scalar.  
The loss is **spectrum-level**: predict the full 750-bin pattern that matches the experimental one.  

The GNN learns end-to-end:  
- which fragments contribute strongly (high intensity scalar)  
- which are spurious (scalar ≈ 0)  
- purely from the gradient signal of the spectrum-level loss

The MAGMa tree constrains **which fragments are structurally plausible** for a given molecule.  
The experimental spectrum teaches **how much each fragment should contribute**.  
Neither source alone is sufficient — both are needed.

In [ ]:
# Demonstrate MAGMa candidate selection on a specific fake experimental peak
import numpy as np

fake_peak_mz = 70.0  # a low-mass peak typical for cyclohexyl fragments

# Step 1: get all (frag, h_shift, iso_shift) entries with isotopes ON
fh, fi, sh, masses_arr, scores_arr = engine.get_frag_masses()

# Step 2: sort by broken-bond score (cheapest first)
sort_idx = np.argsort(scores_arr)
fh, fi, sh, masses_arr, scores_arr = (
    fh[sort_idx],
    fi[sort_idx],
    sh[sort_idx],
    masses_arr[sort_idx],
    scores_arr[sort_idx],
)

# Step 3: mass match — which entries land at this peak?
rounded_obs = round(fake_peak_mz)
matched = np.abs(np.round(masses_arr) - rounded_obs) < 1

# Step 4: keep only minimum-score candidates (up to 5)
min_score = scores_arr[matched].min()
best = np.where(matched & (scores_arr == min_score))[0][:5]

print(f"Fake experimental peak at m/z = {fake_peak_mz}")
print(
    f"All matched entries: {matched.sum()}  →  min_score = {min_score:.2f}  →  {len(best)} best kept"
)
print(
    f"\n{'formula':>20}  {'base_mass':>10}  {'h':>4}  {'iso':>4}  {'score':>7}  {'unit_mz':>8}"
)
calc_local = IsotopePatternCalculator(min_intensity_threshold=0.01)
for i in best:
    fe = engine.frag_to_entry[fh[i]]
    iso = calc_local.calculate_isotope_distribution(
        calc_local.parse_formula(fe.form)
    )
    for iso_s, abund in iso.items():
        if (
            abund >= 0.01
            and round(fe.base_mass + sh[i] * 1.007825 + iso_s) == rounded_obs
        ):
            print(
                f"{fe.form:>20}  {fe.base_mass:>10.3f}  {sh[i]:>4}  {iso_s:>4}  {scores_arr[i]:>7.2f}  {rounded_obs:>8}"
            )

# Draw all unique candidate fragments
unique_hashes = list(dict.fromkeys(fh[b] for b in best))
n = len(unique_hashes)
fig, axes = plt.subplots(1, n, figsize=(min(n * 1.8, 6.5), 2.4))
if n == 1:
    axes = [axes]
for ax, h_val in zip(axes, unique_hashes):
    fe = engine.frag_to_entry[h_val]
    di = engine.get_draw_dict(fe.frag)
    img = Draw.MolToImage(
        di.mol,
        highlightAtoms=di.hatoms,
        highlightBonds=di.hbonds,
        size=(200, 160),
    )
    ax.imshow(img)
    ax.set_title(
        f"{fe.form}\n{fe.base_mass:.1f} Da  score={fe.score:.1f}",
        fontsize=7,
        fontweight="bold",
    )
    ax.axis("off")
plt.suptitle(
    f"All MAGMa candidates for m/z = {fake_peak_mz} (min-score, ≤5)\n"
    "→ ALL stored in DAG; GNN learns from spectrum-level loss which matter",
    fontweight="bold",
)
plt.tight_layout()
plt.savefig("magma_candidates.svg", bbox_inches="tight")
plt.savefig("magma_candidates.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
# Show fragment coverage across the full m/z range for this molecule
# i.e. which integer bins have at least one (frag, h_shift, iso) entry that could explain them
all_covered_bins = set(masses_arr.astype(int))

mz_range = np.arange(1, 230)
covered = np.array([int(mz) in all_covered_bins for mz in mz_range])

fig, ax = plt.subplots(figsize=(6.5, 2.0))
ax.bar(
    mz_range[covered],
    np.ones(covered.sum()),
    width=1.0,
    color="steelblue",
    label="covered by ≥1 fragment",
)
ax.bar(
    mz_range[~covered],
    np.ones((~covered).sum()),
    width=1.0,
    color="#ddd",
    label="no fragment match",
)
ax.axvline(
    engine.full_weight,
    color="red",
    lw=1.5,
    linestyle="--",
    label=f"M = {engine.full_weight:.1f} Da",
)
ax.set_xlabel("m/z", fontweight="bold")
ax.set_yticks([])
ax.set_title(
    f"MAGMa fragment coverage  ({covered.sum()}/{len(mz_range)} bins covered)\n"
    "Any experimental peak falling on a blue bin gets at least one fragment candidate assigned",
    fontweight="bold",
)
ax.legend(fontsize=7)
ax.grid(True, linestyle="--", alpha=0.3, axis="x")
plt.tight_layout()
plt.savefig("magma_coverage.svg", bbox_inches="tight")
plt.savefig("magma_coverage.png", dpi=300, bbox_inches="tight")
plt.show()

print(
    f"\nFragment score range: {engine.frag_to_entry[list(engine.frag_to_entry)[0]].score:.2f} ... "
    f"{max(e.score for e in engine.frag_to_entry.values()):.2f}"
)
print(
    "Score = sum of bond_weight costs along all bonds that cross the fragment boundary."
)
print("Lower score → fewer / cheaper bonds broken → preferred candidate.")
print()
print("Training target summary:")
print(
    "  magma_tree.hdf5 : fragment graphs + is_observed flags (NO intensities)"
)
print("  spectra.hdf5    : raw experimental (mz, intensity) pairs")
print(
    "  FragmentSpecDataset.__getitem__ joins both; inten_targs = binned experimental spectrum"
)
print(
    "  Loss = spectrum-level (cosine/JS/entropy) between GNN output and inten_targs"
)

---
## Summary

### Rings
Atom removal keeps the ring intact as long as both ring neighbours of the removed atom are still present — DFS covers them as one connected component via the intact ring path. Two consecutive removals open the ring.

### IsotopePatternCalculator vs DifferentiableIsotopePatternModule
`IsotopePatternCalculator` is **not used at inference** (`detect_isotope_patterns=False`). It is used in the MAGMa labelling pipeline. At inference/training the GNN uses `DifferentiableIsotopePatternModule` — same binomial math but in PyTorch tensors so gradients flow.

### Fragment selection (max_nodes)
Full BFS always runs first. `max_nodes` prunes afterwards by `(tree_depth, max_broken)` ascending — keeps shallowest, cheapest fragments.

### Predicted intensity in full-enumeration mode
Fragments from BFS. GNN outputs one scalar per (fragment, h_shift). `DifferentiableIsotopePatternModule` spreads each scalar to M+0, M+2, … bins. `scatter_add_` sums all contributions per bin. Loss vs experimental spectrum.

### Same-mass collision
Bin value = sum of all contributors. `frag_hash` label = highest-intensity contributor (visualisation only).

### MAGMa labelling + intensity targets (the key insight)
**The DAG stores no intensities** — `intens` is always zeros in `magma_tree.hdf5`.  
The pipeline does three things:
1. **Candidate assignment**: for each experimental peak, find all (frag, h_shift, iso) entries within ±0.5 Da, keep those with the minimum broken-bond score (up to 5). Multiple candidates at the same score are all kept — ambiguity is unresolved.
2. **Tree pruning**: trace ancestors of all matched fragments to root, then greedy-prune to a minimal connected subtree.
3. **Structure only**: the saved DAG tells the GNN *which fragments are plausible*. The experimental spectrum (`spectra.hdf5`) provides the actual intensity target, loaded at `__getitem__` time and binned into 750 bins.

The GNN is **never given per-fragment intensity labels**. It learns end-to-end from a spectrum-level loss (cosine/JS/entropy) to assign scalars to fragments such that their scatter-added, isotope-expanded contributions match the experimental pattern.